In [1]:
from astropy.io import fits
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
import pandas as pd

In [2]:
DP = DP()
df_old, _, _, _ = DP.extract_fits_data('./catalogs/0315_dps_test.fits')
df_new, _, _, _ = DP.extract_fits_data('./catalogs/0428_dps_test.fits')

In [3]:
df_old = df_old[df_old['BPT_2COMP'] != 512].copy()
df_new = df_new[df_new['BPT_2COMP'] != 512].copy()

In [4]:
print(len(df_old), len(df_new))

13992 13706


In [5]:
df_old_wo_new = df_old[~df_old['TARGETID'].isin(df_new['TARGETID'])].copy()

In [6]:
df_old_wo_new.head(10)

,TARGETID,RA,DEC,Z,LOGM,LOGSFR,DV_R,DV_L,SIGMA_R,SIGMA_L,...,NII6583_SNR,SII6716_SNR,SII6731_SNR,BPT_1COMP,BPT_2COMP_L,BPT_2COMP_R,BPT_2COMP,LOGSFR_1COMP,LOGSFR_2COMP,dp_count
7,39627739384253866,177.401154,-1.923410,0.322626,10.556721,1.160900,7.521114e+01,-84.288429,63.610744,50.902027,...,5.241505,-1.000000,-1.000000,1,1,1,2,0.266869,-0.488258,2
10,39627739388446113,177.554276,-1.971866,0.083630,10.489296,0.985447,7.153632e+01,-36.476711,34.187515,53.084030,...,44.942814,13.280462,11.475150,4,4,1,5,0.679293,0.704114,3
12,39627739388450140,177.711121,-1.988391,0.175282,10.060081,0.080345,3.577178e+01,-79.494247,46.249466,47.375790,...,4.452712,3.703881,-1.000000,1,1,1,2,0.157859,-0.625933,1
29,39627739401030023,178.357407,-1.959469,0.163351,9.867171,0.379196,1.314868e+01,-53.302532,55.138092,0.001000,...,11.469440,3.888617,3.690983,4,16,4,20,0.044481,-0.204860,1
38,39627739405227788,178.747360,-1.990688,0.176721,9.596325,0.350646,1.107697e-16,-2.226653,60.150063,18.940838,...,17.072380,13.002821,11.623968,1,1,1,2,0.219385,1.621846,1
71,39627739430390772,180.124969,-1.893296,0.271685,10.405784,1.285207,5.903730e+01,-81.712624,78.753586,71.982971,...,11.206850,5.648541,3.983813,4,4,4,8,0.788366,0.356449,2
82,39627739442972275,180.798370,-1.959218,0.154199,10.272751,0.636115,5.078999e+01,-19.310577,0.001037,75.435158,...,8.419136,-1.000000,-1.000000,1,1,1,2,0.541262,0.416162,2
98,39627739459749222,181.793259,-1.905957,0.130609,9.485508,-0.521147,6.808779e+01,-17.489141,15.438609,36.235676,...,9.484408,3.486811,-1.000000,1,1,1,2,-0.396238,-0.700049,2
104,39627739468138950,182.351044,-1.938127,0.158024,10.552460,1.079776,3.217157e+01,-122.639954,71.729027,35.939972,...,6.339388,3.428163,3.197396,1,1,1,2,0.367350,0.053261,1
111,39627739472332989,182.583069,-1.956027,0.319307,10.674972,-0.145007,6.362466e+01,-76.088921,62.732109,42.300369,...,4.049060,-1.000000,-1.000000,4,4,4,8,-0.050068,-0.944838,1


In [7]:
classification_map = {
    1: 'SF', 4: 'COMP', 16: 'AGN', 64: 'LINER', 256: 'unclassified',
    2: 'double SF', 8: 'double COMP', 32: 'double AGN', 128: 'double LINER',
    5: 'SF+COMP', 17: 'SF+AGN', 65: 'SF+LINER',
    20: 'COMP+AGN', 68: 'COMP+LINER', 80: 'AGN+LINER',
    257: 'SF+uncertain', 260: 'COMP+uncertain', 272: 'AGN+uncertain', 320: 'LINER+uncertain',
    512: 'unclassified'
}

In [8]:
df_old['BPT_2COMP'] = df_old['BPT_2COMP'].map(classification_map)
df_new['BPT_2COMP'] = df_new['BPT_2COMP'].map(classification_map)

In [9]:
print(df_old['BPT_2COMP'].value_counts(normalize=True)*100)
print(df_new['BPT_2COMP'].value_counts(normalize=True)*100)


BPT_2COMP
double SF         73.763579
SF+COMP           11.828188
SF+AGN             5.974843
double COMP        3.051744
COMP+AGN           2.565752
double AGN         2.208405
AGN+LINER          0.235849
COMP+LINER         0.157233
SF+LINER           0.092910
AGN+uncertain      0.057176
COMP+uncertain     0.042882
SF+uncertain       0.014294
double LINER       0.007147
Name: proportion, dtype: float64
BPT_2COMP
double SF         67.423026
SF+COMP           16.926893
SF+AGN             5.464760
double COMP        5.333431
COMP+AGN           2.254487
double AGN         1.495695
COMP+LINER         0.379396
AGN+LINER          0.328323
SF+LINER           0.269955
COMP+uncertain     0.065665
AGN+uncertain      0.058369
Name: proportion, dtype: float64
